In [1]:
import pandas as pd
import re
from ete3 import Tree, TreeStyle, NodeStyle, faces, AttrFace
import matplotlib.pyplot as plt
from ete3 import RectFace

In [2]:
#----------------------------------
# EXTRACT GENOME
data = pd.read_csv("../data/df_filtered.csv")

def extract_genome(x):
    if pd.isna(x):
        return None
    first = x.split(",")[0]
    parts = first.split("_")
    return "_".join(parts[:2])  # GCA + accession.version

data["genome"] = data["target"].apply(extract_genome)

In [3]:
#----------------------------------
# DATA PREP FOR TREE

out = data[["genome", "SQ_pathway"]].copy()

out["genome"] = "MAG_" + out["genome"] # добавляем MAG_ чтобы совпало с GTDB

out = out.rename(columns={"genome": "user_genome"}) # переименовываем под merge

wide = (
    out.assign(value=1)  # добавляем столбец для заполнения
       .pivot_table(index="user_genome",
                    columns="SQ_pathway",
                    values="value",
                    fill_value=0)
       .reset_index()
)

#----------------------------------
# INDEX

df = pd.read_csv("../data/classify/gtdbtk.bac120.summary.tsv", sep="\t")
df = df.merge(wide, on="user_genome", how="left")

df["closest_genome_reference"] = df["closest_genome_reference"].replace("N/A", pd.NA)
df["tree_rep"] = df["closest_genome_reference"].fillna(df["user_genome"])

def extract_id(x):
    if pd.isna(x):
        return None
    m = re.search(r"(\d+\.\d+)", x)
    return m.group(1) if m else x

def get_order(x):
    if pd.isna(x):
        return None
    for part in x.split(";"):
        if part.startswith("o__"):
            return part.replace("o__", "")
    return None

df["gid"] = df["tree_rep"].apply(extract_id)
df["order"] = df["classification"].apply(get_order)

my_ids = set(df["gid"])
id_to_order = dict(zip(df["gid"], df["order"]))

unique_orders = list(dict.fromkeys(df["order"].dropna()))

In [4]:
#----------------------------------
# LOAD TREE + PROCESSING

t = Tree("../data/classify/gtdbtk.bac120.classify.tree.4.tree", format=1, quoted_node_names=True)

for leaf in t.iter_leaves(): # map IDs
    leaf.add_feature("gid", extract_id(leaf.name))

for leaf in list(t.iter_leaves()): # filter by IDs
    if leaf.gid not in my_ids:
        leaf.delete()

In [6]:
#----------------------------------
# COLOR MAP

unique_orders = list(dict.fromkeys(id_to_order.values()))
unique_orders = [o for o in unique_orders if o is not None]

palette = plt.cm.tab20.colors

order_to_color = {
    "Acidaminococcales": "#42855a",
    "Christensenellales": "#ffa7a7",
    "Eubacteriales": "#0b731e",
    "Lachnospirales": "#83adff",
    "Monoglobales": "#6d54b8",
    "Oscillospirales": "#f9b775",
    "Peptostreptococcales": "#bfe085",
    "RUG12999": "#c38046",
    "Selenomonadales": "#7bbd8d",
    "UBA1212": "#5254a3",
    "UBA1381": "#ba5aa4",
    "UBA7702": "#4b7b78"
}

pathways = ["sulfo-ASMO", "sulfo-ED", "sulfo-EMP", "sulfo-TAL", "sulfo-TK"]

pathway_colors = {
    'sulfo- ASMO': '#d67769',
    'sulfo-ED': '#ffd700',
    'sulfo-EMP': "#2CAEA1",
    'sulfo-TAL': '#df81b5',
    'sulfo-TK': '#993399'
}

gid_to_row = df.set_index("gid")

#----------------------------------
# LAYOUT

def layout(node):
    if not node.is_leaf():
        return

    gid = getattr(node, "gid", None)
    if gid is None:
        return

    order = id_to_order.get(gid)

    # node style
    if order is not None:
        color_hex = order_to_color.get(order, "#000000")

        if not isinstance(color_hex, str):
            color_hex = "#000000"

        nstyle = NodeStyle()
        nstyle["fgcolor"] = "black"
        nstyle["size"] = 5
        nstyle["bgcolor"] = color_hex
        nstyle["shape"] = "circle"
        node.set_style(nstyle)

    # name face
    faces.add_face_to_node(
        AttrFace("name", fgcolor="black", fsize=10),
        node,
        column=0,
        position="aligned"
    )

    # pathway faces
    if gid in gid_to_row.index:
        row = gid_to_row.loc[gid]

        col_offset = 1

        for pathway in pathways:
            if pathway in row and row[pathway] == 1:
                color = pathway_colors.get(pathway, "#000000")

                rect = RectFace(
                    width=20,    
                    height=20,    
                    fgcolor=color,
                    bgcolor=color
                )

                rect.border.width = 0

                rect.margin_top = 1
                rect.margin_bottom = 1

                faces.add_face_to_node(
                    rect,
                    node,
                    column=col_offset,
                    position="aligned"
                )

                col_offset += 1

#----------------------------------
# TREE STYLE 

ts = TreeStyle() 
ts.mode = "c" 
ts.show_leaf_name = False 
ts.scale = 720 
ts.layout_fn = layout

t.render("../pictures/tree/my_tree.png", tree_style=ts, w=4000)

{'nodes': [[1996.3495581947595,
   1998.1177370998269,
   2005.2018857234218,
   2006.9700646284891,
   0,
   None],
  [2060.980902825111,
   2210.0696246674156,
   2069.8332303537736,
   2218.9219521960777,
   1,
   None],
  [2257.2149287540215,
   2007.746857191124,
   2264.611029612511,
   2015.1429580496144,
   2,
   None],
  [2320.8054254596896,
   2000.204989355026,
   2327.9799627110474,
   2007.379526606384,
   3,
   None],
  [2450.2824032112208,
   1994.680911294412,
   2460.920580622397,
   2005.3190887055882,
   4,
   None],
  [2420.5667921689405,
   2004.519746724798,
   2431.450755912856,
   2015.4037104687138,
   5,
   None],
  [2294.724992859847,
   2018.922155941254,
   2302.335014659755,
   2026.5321777411616,
   6,
   None],
  [2316.9351583398566,
   2015.017782220041,
   2324.4294569002395,
   2022.5120807804242,
   7,
   None],
  [2377.617212883421,
   2012.3683155787387,
   2388.7410139959734,
   2023.4921166912907,
   8,
   None],
  [2376.9764520397166,
   2021.20

In [7]:
# tree legend

def save_tree_legend(order_to_color, pathway_colors, filename="tree_legend.png"):

    n_orders = len(order_to_color)
    n_pathways = len(pathway_colors)

    fig_height = max(3, (n_orders + n_pathways) * 0.35)
    fig, ax = plt.subplots(figsize=(6, fig_height))

    ax.axis("off")

    y = 0

    # ORDERS SECTION

    orders_start_y = y

    for order in reversed(list(order_to_color.keys())):

        color_hex = order_to_color[order]

        if not isinstance(color_hex, str):
            color_hex = "#000000"

        ax.add_patch(
            plt.Rectangle((0, y), 0.5, 0.5, color=color_hex)
        )

        ax.text(0.7, y + 0.1, order, fontsize=12, va="bottom")
        y += 0.7

    ax.text(
        0,
        orders_start_y + (y - orders_start_y) + 0.3,
        "Orders",
        fontsize=12,
        fontweight="bold"
    )

    y += 1
    
    # PATHWAYS SECTION
    
    pathways_start_y = y

    for pathway, color_hex in reversed(list(pathway_colors.items())):

        if not isinstance(color_hex, str):
            color_hex = "#000000"

        ax.add_patch(
            plt.Rectangle((0, y), 0.5, 0.5, color=color_hex)
        )

        ax.text(0.7, y + 0.1, pathway, fontsize=12, va="bottom")
        y += 0.7

    ax.text(
        0,
        pathways_start_y + (y - pathways_start_y) + 0.3,
        "Pathways",
        fontsize=12,
        fontweight="bold"
    )

    # LIMITS

    ax.set_xlim(0, 6)
    ax.set_ylim(0, y + 1)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()

save_tree_legend(order_to_color, pathway_colors, "../pictures/tree/tree_legend.png")